In [1]:
import pandas as pd
import numpy as np
import scanpy as sc
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, Dataset
from scipy.spatial.distance import cdist
import torch.nn.functional as F

from model.mil import MIL, MILAggregator
from model.dataset import BagsDataset


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [3]:
adata = sc.read_h5ad('/work/DPDS/s439765/example.h5ad')
adata.obs

,in_tissue,array_row,array_col,n_counts,X,Y,tcr
AAACAAGTATCTCCCA-1,1,50,102,5625.0,5000,10200,0
AAACACCAATAACTGC-1,1,59,19,9159.0,5900,1900,0
AAACAGAGCGACTCCT-1,1,14,94,2660.0,1400,9400,1
AAACAGGGTCTATATT-1,1,47,13,9020.0,4700,1300,1
AAACAGTGTTCCTGGG-1,1,73,43,9714.0,7300,4300,0
...,...,...,...,...,...,...,...
TTGTTGTGTGTCAAGA-1,1,31,77,7260.0,3100,7700,0
TTGTTTCACATCCAGG-1,1,58,42,7510.0,5800,4200,1
TTGTTTCATTAGTCTA-1,1,60,30,5742.0,6000,3000,0
TTGTTTCCATACAACT-1,1,45,27,20118.0,4500,2700,1


In [4]:
class BagsDataset(Dataset):
    def __init__(self, adata, radius=50, output_csv='bags.csv'):
        self.bags = self.create_bags(adata, radius, output_csv)

    def __len__(self):
        return len(self.bags)

    def __getitem__(self, idx):
        bag = self.bags[idx]
        instances = bag['instances']
        label = bag['label']
        return instances, label

    def create_bags(self, adata, radius, output_csv):
        spatial_coords_x = adata.obs['X']
        spatial_coords_y = adata.obs['Y']
        spatial_coords = np.array(list(zip(spatial_coords_x, spatial_coords_y)))

        gene_expression = adata.X
        labels = adata.obs['tcr'].values
        barcodes = adata.obs.index.values
        bags = {}

        dist_matrix = cdist(spatial_coords, spatial_coords, metric='euclidean')

        csv_data = []

        for i in range(len(spatial_coords)):
            in_circle = np.where(dist_matrix[i] <= radius)[0]

            bag_data = gene_expression[in_circle] # Convert sparse matrix to dense
            bag_label = labels[i]
            bag_barcodes = barcodes[in_circle]

            bags[i] = {
                'instances': bag_data,
                'label': bag_label
            }

            for barcode in bag_barcodes:
                csv_data.append([i, barcode, bag_label])

        df = pd.DataFrame(csv_data, columns=['bag_id', 'barcode', 'label'])
        df.to_csv(output_csv, index=False)

        return bags

def custom_collate_fn(batch):
    # Custom collate function to handle bags with variable number of instances
    instances, labels = zip(*batch)
    return instances, torch.tensor(labels[0], dtype=torch.float32).view(1)

In [5]:
dataset = BagsDataset(adata, radius=200)

In [6]:
total_size = len(dataset)
train_size = int(0.8 * total_size)
val_size = int(0.2 * total_size)
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])


In [7]:
def custom_collate_fn(batch):
    # Custom collate function to handle bags with variable number of instances
    instances, labels = zip(*batch)
    return instances, torch.tensor(labels[0], dtype=torch.float32).view(1)

In [8]:

train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=custom_collate_fn)
val_dataloader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=custom_collate_fn)

In [9]:

dataloader = DataLoader(dataset, batch_size=16, shuffle=True, collate_fn=custom_collate_fn)
len(dataloader)

165

In [10]:
print(device)

cuda


In [11]:
from torchsummary import summary   
mil = MIL(1).to(device)
summary(mil, (1,))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Linear-1                   [-1, 64]             128
            Linear-2                   [-1, 32]           2,080
            Linear-3                    [-1, 1]              33
Total params: 2,241
Trainable params: 2,241
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.01
Estimated Total Size (MB): 0.01
----------------------------------------------------------------


In [12]:
model= MILAggregator(adata.n_vars)
model.to(device)


MILAggregator(
  (mil): MIL(
    (fc1): Linear(in_features=19379, out_features=64, bias=True)
    (fc2): Linear(in_features=64, out_features=32, bias=True)
    (fc3): Linear(in_features=32, out_features=1, bias=True)
  )
  (attention): Attention(
    (feature_extractor_part1): Sequential(
      (0): Linear(in_features=1, out_features=128, bias=True)
      (1): ReLU()
      (2): Linear(in_features=128, out_features=500, bias=True)
      (3): ReLU()
    )
    (attention): Sequential(
      (0): Linear(in_features=500, out_features=128, bias=True)
      (1): Tanh()
      (2): Linear(in_features=128, out_features=1, bias=True)
    )
    (classifier): Sequential(
      (0): Linear(in_features=500, out_features=1, bias=True)
      (1): Sigmoid()
    )
  )
)

In [13]:
model = MILAggregator(adata.n_vars).to(device)

In [14]:
criterion = nn.BCELoss().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [15]:
def evaluate_model(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    with torch.no_grad():
        for instances, label in dataloader:
            instances = [torch.tensor(instance, dtype=torch.float32).to(device) for instance in instances].to(device)
            label = torch.tensor(label, dtype=torch.float32).to(device)

            instance_outputs = [model(instance) for instance in instances]
            instance_outputs = torch.cat(instance_outputs, dim=0)
            bag_output = torch.sum(instance_outputs, dim=0, keepdim=True)

            label = label.view_as(bag_output)

            loss = criterion(bag_output, label)
            running_loss += loss.item()

            if bag_output.shape == label.shape:
                preds = torch.round(torch.sigmoid(bag_output)) 
                correct_predictions += (preds == label).sum().item()
                total_samples += label.size(0)

    avg_loss = running_loss / len(dataloader)
    accuracy = correct_predictions / total_samples

    return avg_loss, accuracy



In [17]:

num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for batch_idx, (instances, label) in enumerate(train_dataloader):
        instances = [torch.tensor(instance, dtype=torch.float32).to(device) for instance in instances]
        label = torch.tensor(label, dtype=torch.float32).to(device)

        optimizer.zero_grad()

        instance_outputs = [model(instance) for instance in instances]
        instance_outputs = torch.cat(instance_outputs, dim=0)

        bag_output = torch.sum(instance_outputs, dim=0, keepdim=True)
        
        label = label.view_as(bag_output)

        loss = criterion(bag_output, label)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        if (batch_idx + 1) % 10 == 0:
            print(f'Epoch [{epoch+1}/{10}], Step [{batch_idx+1}/{len(train_dataloader)}], Loss: {loss.item():.4f}')

    print(f'Epoch [{epoch+1}/{10}], Loss: {running_loss / len(train_dataloader):.4f}')

    val_loss, val_accuracy = evaluate_model(model, val_dataloader, criterion, device)
    print(f'Epoch [{epoch+1}/{10}], Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_accuracy:.4f}')

print("Finished Training")


/tmp/ipykernel_69161/3841276236.py:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  label = torch.tensor(label, dtype=torch.float32).to(device)
../aten/src/ATen/native/cuda/Loss.cu:94: operator(): block: [0,0,0], thread: [0,0,0] Assertion `input_val >= zero && input_val <= one` failed.


RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1.
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [15]:
"""
withoud validation
"""

for epoch in range(10):
    model.train()
    running_loss = 0.0

    for batch_idx, (instances, label) in enumerate(dataloader):
        instances = [torch.tensor(instance, dtype=torch.float32).to(device) for instance in instances]
        label = torch.tensor(label, dtype=torch.float32).to(device)

        optimizer.zero_grad()

        
        instance_outputs = [model(instance) for instance in instances]
        instance_outputs = torch.cat(instance_outputs, dim=0)

    
        bag_output = torch.mean(instance_outputs, dim=0, keepdim=True)
        
        label = label.view_as(bag_output)

        loss = criterion(bag_output, label)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        if (batch_idx + 1) % 10 == 0:
            print(f'Epoch [{epoch+1}/{10}], Step [{batch_idx+1}/{len(dataloader)}], Loss: {loss.item():.4f}')

    print(f'Epoch [{epoch+1}/{10}], Loss: {running_loss / len(dataloader):.4f}')

print("Finished Training")

/tmp/ipykernel_4671/3845836986.py:7: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  label = torch.tensor(target, dtype=torch.float32).to(device)


Epoch [1/10], Step [10/165], Loss: 0.6788
Epoch [1/10], Step [20/165], Loss: 0.6693
Epoch [1/10], Step [30/165], Loss: 0.6669
Epoch [1/10], Step [40/165], Loss: 0.6542
Epoch [1/10], Step [50/165], Loss: 0.6443
Epoch [1/10], Step [60/165], Loss: 0.6370
Epoch [1/10], Step [70/165], Loss: 0.6294
Epoch [1/10], Step [80/165], Loss: 0.6224
Epoch [1/10], Step [90/165], Loss: 0.6156
Epoch [1/10], Step [100/165], Loss: 0.6088
Epoch [1/10], Step [110/165], Loss: 0.6028
Epoch [1/10], Step [120/165], Loss: 0.5962
Epoch [1/10], Step [130/165], Loss: 0.5937
Epoch [1/10], Step [140/165], Loss: 0.5841
Epoch [1/10], Step [150/165], Loss: 0.5779
Epoch [1/10], Step [160/165], Loss: 0.5716
Epoch [1/10], Loss: 0.6234
Epoch [2/10], Step [10/165], Loss: 0.5606
Epoch [2/10], Step [20/165], Loss: 0.5555
Epoch [2/10], Step [30/165], Loss: 0.5446
Epoch [2/10], Step [40/165], Loss: 0.5357
Epoch [2/10], Step [50/165], Loss: 0.5261
Epoch [2/10], Step [60/165], Loss: 0.5158
Epoch [2/10], Step [70/165], Loss: 0.5048


#train aggregator——expired 

num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for i, (instances, label) in enumerate(dataloader):
        instances = [torch.tensor(np.array(instance), dtype=torch.float32).to(device) for instance in instances[0]]
        label = torch.tensor(label, dtype=torch.float32).to(device)

        optimizer.zero_grad()

       
        output = model(instances)


        loss = criterion(output, label)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        if i % 10 == 9:  
            print(f"Epoch [{epoch+1}/{num_epochs}], Step [{i+1}], Loss: {loss.item():.4f}")

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss / len(dataloader):.4f}")

print("Finished Training")


In [19]:
torch.save(model.state_dict(), 'mil_aggregator.pth')

In [21]:
test_dataloader = DataLoader(dataset, batch_size=1, shuffle=True, collate_fn=custom_collate_fn)

In [24]:
evaluate_model(model, list(test_dataloader)[0:1], device)

Accuracy of the model on the entire dataset: 0.0%
Data point 0, Bag 0 weights: [[1.1829333e-06 1.4285684e-01 1.4285684e-01 1.4285684e-01 1.4285684e-01
  1.0520686e-06 1.4285684e-01 1.4285684e-01 1.4285684e-01]]


/tmp/ipykernel_52482/208287778.py:10: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  label = torch.tensor(label, dtype=torch.float32).to(device)


In [22]:
def predict(model, adata, device, radius=50, batch_size=1):
    model.eval()
    dataset = BagsDataset(adata, radius)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=custom_collate_fn)
    
    predictions = []

    with torch.no_grad():
        for instances, _ in dataloader:
            instances = [torch.tensor(instance, dtype=torch.float32).to(device) for instance in instances[0]]
            
            instance_outputs = [model(instance) for instance in instances]
            instance_outputs = torch.cat(instance_outputs, dim=0)

            bag_output = torch.mean(instance_outputs, dim=0, keepdim=True)
            predictions.append(bag_output.cpu().numpy().flatten())

    predictions = np.array(predictions)
    adata.obs['tcr_predict'] = predictions
    return adata

In [23]:
adata = predict(model, adata, device, radius=200, batch_size=1)

In [24]:
adata.obs

,in_tissue,array_row,array_col,n_counts,X,Y,tcr,tcr_predict
AAACAAGTATCTCCCA-1,1,50,102,5625.0,5000,10200,0,0.497867
AAACACCAATAACTGC-1,1,59,19,9159.0,5900,1900,1,0.503233
AAACAGAGCGACTCCT-1,1,14,94,2660.0,1400,9400,0,0.508804
AAACAGGGTCTATATT-1,1,47,13,9020.0,4700,1300,0,0.507334
AAACAGTGTTCCTGGG-1,1,73,43,9714.0,7300,4300,1,0.507056
...,...,...,...,...,...,...,...,...
TTGTTGTGTGTCAAGA-1,1,31,77,7260.0,3100,7700,1,0.500590
TTGTTTCACATCCAGG-1,1,58,42,7510.0,5800,4200,1,0.508704
TTGTTTCATTAGTCTA-1,1,60,30,5742.0,6000,3000,1,0.508035
TTGTTTCCATACAACT-1,1,45,27,20118.0,4500,2700,0,0.508707


In [46]:
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader, Dataset
from scipy.spatial.distance import cdist

class BagsDataset(Dataset):
    def __init__(self, adata, radius=50, output_csv='bags.csv'):
        self.bags = self.create_bags(adata, radius, output_csv)

    def __len__(self):
        return len(self.bags)

    def __getitem__(self, idx):
        bag = self.bags[idx]
        instances = bag['instances']
        label = bag['label']
        return instances, label

    def create_bags(self, adata, radius, output_csv):
        spatial_coords_x = adata.obs['X']
        spatial_coords_y = adata.obs['Y']
        spatial_coords = np.array(list(zip(spatial_coords_x, spatial_coords_y)))

        gene_expression = adata.X
        labels = adata.obs['tcr'].values
        barcodes = adata.obs.index.values
        bags = {}

        dist_matrix = cdist(spatial_coords, spatial_coords, metric='euclidean')

        csv_data = []

        for i in range(len(spatial_coords)):
            in_circle = np.where(dist_matrix[i] <= radius)[0]

            bag_data = gene_expression[in_circle].todense()  # Convert sparse matrix to dense
            bag_label = labels[i]
            bag_barcodes = barcodes[in_circle]

            bags[i] = {
                'instances': bag_data,
                'label': bag_label
            }

            for barcode in bag_barcodes:
                csv_data.append([i, barcode, bag_label])

        df = pd.DataFrame(csv_data, columns=['bag_id', 'barcode', 'label'])
        df.to_csv(output_csv, index=False)

        return bags

def custom_collate_fn(batch):
    # Custom collate function to handle bags with variable number of instances
    instances, labels = zip(*batch)
    return instances, torch.tensor(labels[0], dtype=torch.float32).view(1)


radius = 200
output_csv = 'bags.csv'
dataset = BagsDataset(adata, radius, output_csv)


In [43]:
adata.obs.index.values

array(['AAACAAGTATCTCCCA-1', 'AAACACCAATAACTGC-1', 'AAACAGAGCGACTCCT-1',
       ..., 'TTGTTTCATTAGTCTA-1', 'TTGTTTCCATACAACT-1',
       'TTGTTTGTATTACACG-1'], dtype=object)